# Chain of Thought (CoT) Prompting Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/cot-prompting.ipynb)

## Overview

This tutorial introduces Chain of Thought (CoT) prompting, a powerful technique that encourages AI models to break down complex problems into step-by-step reasoning. We use **Qwen3-8B**, which has **native thinking mode** — the model can reason internally in a `<think>...</think>` block before answering, making CoT a first-class capability rather than just a prompting trick.

## Motivation

As AI language models become more advanced, there's an increasing need to guide them towards producing more transparent, logical, and verifiable outputs. Qwen3's native thinking mode (`enable_thinking=True`) lets us inspect the model's internal reasoning trace, compare thinking vs. non-thinking output, and combine classic CoT prompt techniques with built-in reasoning.

## Key Components

1. **Zero-Shot CoT**: The simplest approach — just append "Let's think step by step" to any prompt.
2. **Few-Shot CoT**: Provide worked examples with step-by-step reasoning before the actual question.
3. **Structured CoT**: Use numbered steps and explicit formatting to guide the model's reasoning.
4. **Comparative Analysis**: Examining the differences between standard and CoT prompting on the same problem.
5. **Problem-Solving Applications**: Applying CoT to complex logical reasoning tasks.

## Method Details

The tutorial will guide learners through the following methods:

1. **Setting up the environment**: Loading an open-source model locally in Google Colab with 4-bit quantization.

2. **Zero-Shot CoT**: The simplest and often surprisingly effective CoT technique.

3. **Few-Shot CoT**: Providing worked examples to teach the model the reasoning pattern.

4. **Structured CoT**: Using explicit numbered-step formats for systematic reasoning.

5. **Practical Applications**: Applying CoT prompting to logical reasoning puzzles and multi-step math problems.

## Conclusion

By the end of this tutorial, learners will have a solid understanding of Chain of Thought prompting and its applications. They will be equipped with practical skills to implement CoT techniques in various scenarios, improving the quality and interpretability of AI-generated responses. This knowledge will be valuable for anyone working with large language models, from developers and researchers to business analysts and decision-makers relying on AI-powered insights.

## Why Intermediate Steps Help: The Error Decomposition Argument

### The Mathematical Intuition

Consider a complex problem that requires reasoning across `N` steps. If a model attempts to produce the final answer in **one shot**, it must make a single, high-difficulty prediction — essentially compressing `N` reasoning steps into one token-generation decision. The probability of getting this right is some value `p_direct`, which is often low for hard problems.

Now consider decomposing the problem into `N` intermediate steps, each of difficulty `d_i`. If each step has success probability `p_i`, the overall success probability is:

```
P(all steps correct) = p_1 × p_2 × ... × p_N
```

At first glance, multiplying `N` probabilities seems worse (errors compound!). But here's the key insight: **each `p_i` is much closer to 1.0** because each individual step is within the model's capability. For example:

| Approach | Steps | Per-step accuracy | Overall accuracy |
|----------|-------|-------------------|------------------|
| Direct | 1 | 30% | **30%** |
| CoT (5 steps) | 5 | 92% | **66%** |
| CoT (5 steps) | 5 | 95% | **77%** |
| CoT (5 steps) | 5 | 98% | **90%** |

Even with error compounding across 5 steps, CoT wins when each step is easy enough for the model.

### The Maze Analogy

Think of it like navigating a maze:
- **Direct answering** = teleporting to where you *think* the exit is. If the maze is complex, you'll probably land in a wall.
- **Chain of Thought** = walking step by step, checking at each intersection. Each individual turn decision is easy, and you can see the walls around you.

The "walls you can see" are the intermediate results — once the model writes "Step 1: distance = 150 km", that text becomes part of the attention context for Step 2. The model is literally reading its own work.

In [ ]:
# Demonstrate: Direct answer vs. step-by-step on a multi-step math problem
# This problem requires multiple operations — the kind where direct answers often go wrong.

multi_step_problem = (
    "A store sells apples at $1.20 each and oranges at $0.80 each. "
    "Maria buys 5 apples and 8 oranges, pays with a $20 bill, and gets change. "
    "She then gives half her change to her brother. "
    "How much money does Maria have left from the change?"
)
# Correct answer: 5×1.20 + 8×0.80 = 6.00 + 6.40 = 12.40.
# Change = 20 - 12.40 = 7.60.  Half = 3.80.  Maria keeps $3.80.

# --- Direct answer (no CoT, no thinking) ---
direct_msgs = [{"role": "user", "content": f"Answer with just the final number: {multi_step_problem}"}]
direct_answer = generate(direct_msgs, temperature=0.1, thinking=False)
print("=== DIRECT ANSWER (no reasoning) ===")
print(f"Answer: {direct_answer}")

# --- Step-by-step with native thinking ---
cot_msgs = [{"role": "user", "content": f"{multi_step_problem}\nSolve this step by step."}]
thought, cot_answer = generate(cot_msgs, temperature=0.6)
print("\n=== CHAIN OF THOUGHT (step-by-step) ===")
print(f"Thinking trace ({len(thought.split())} words):")
print(thought[:600] if thought else "(empty)")
print(f"\nFinal answer: {cot_answer}")
print(f"\nCorrect answer: $3.80")

### Connection to Attention: "Reading Your Own Work"

Why does writing out intermediate steps help a neural network? The answer lies in the **attention mechanism**.

When the model generates token `t_i`, it attends to **all previous tokens** — including tokens it generated itself in earlier steps. This means:

1. **Step 1 output** becomes part of the context for Step 2
2. **Step 2 output** (which already incorporated Step 1) becomes context for Step 3
3. And so on...

This creates a form of **external working memory**. Without CoT, the model must hold all intermediate results in its hidden state (fixed-size vector). With CoT, intermediate results are written out as text and can be re-read through attention — effectively unlimited working memory.

This is analogous to how humans solve math problems: we write down intermediate results on paper rather than keeping everything in our heads. The paper is our external memory, and our eyes reading it back is our "attention mechanism."

## Setup

We load the **Qwen3-8B** model locally using HuggingFace Transformers with 4-bit quantization so it fits in a free Colab GPU. The model weights are cached to Google Drive to avoid re-downloading on every session.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.6, do_sample=True, thinking=True):
    """Generate a response, optionally with Qwen3's native thinking mode.
    Returns (thinking_text, answer) when thinking=True, otherwise just the answer string."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens, do_sample=do_sample, top_k=20,
        temperature=temperature if do_sample else None,
        top_p=0.95 if thinking else 0.8,
    )
    output_ids = model.generate(**inputs, **gen_kwargs)
    token_ids = output_ids[0][inputs.input_ids.shape[1]:].tolist()
    if thinking:
        # Parse <think>...</think> block (token 151668 = </think>)
        try:
            idx = len(token_ids) - token_ids[::-1].index(151668)
        except ValueError:
            idx = 0
        thought = tokenizer.decode(token_ids[:idx], skip_special_tokens=True).strip()
        answer  = tokenizer.decode(token_ids[idx:], skip_special_tokens=True).strip()
        return thought, answer
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()

## 1 · Zero-Shot CoT — "Let's think step by step"

The simplest form of CoT prompting: just append **"Let's think step by step"** to any question. This single phrase can dramatically improve reasoning accuracy, even without any worked examples.

In [ ]:
question = "If a train travels 120 km in 2 hours, what is its average speed in km/h?"

# Standard prompt — thinking=False (fast, no reasoning trace)
standard = [{"role": "user", "content": f"Answer the following question concisely: {question}"}]
print("Standard:", generate(standard, temperature=0.1, thinking=False))

# Zero-shot CoT — let Qwen3 think natively
cot_zero = [{"role": "user", "content": f"{question} Let's think step by step."}]
thought, answer = generate(cot_zero, temperature=0.6)
print("\n--- Qwen3 internal reasoning ---")
print(thought[:500] if thought else '(empty)')
print("\n--- Final answer ---")
print(answer)

## 2 · System-Guided CoT

Another zero-shot variant: use the **system message** to instruct the model to always show its reasoning before giving the final answer. This keeps the user prompt clean.

In [ ]:
question = "If a train travels 120 km in 2 hours, what is its average speed in km/h?"

cot_system = [
    {"role": "system", "content": "You are a math tutor. Always show your reasoning step by step before giving the final answer."},
    {"role": "user", "content": question}
]
print("System-guided CoT:")
thought, answer = generate(cot_system, temperature=0.6)
print("Thinking:", thought[:300] if thought else '(none)')
print("Answer:", answer)

## Native Thinking vs. Prompt-Induced CoT

Qwen3 offers two fundamentally different ways to get chain-of-thought reasoning. Let's compare them head-to-head on the same problem to understand the differences.

| Mode | How it works | Quality | Token cost |
|------|-------------|---------|------------|
| **Direct** (`thinking=False`, no CoT prompt) | Model directly predicts the answer | Baseline | Lowest |
| **Prompt-induced CoT** (`thinking=False`, CoT prompt) | Prompt asks for step-by-step reasoning in the *visible* output | Good | Medium |
| **Native thinking** (`thinking=True`) | Model reasons in a hidden `<think>` block, then gives a clean answer | Best | Highest |

In [ ]:
# Same problem, three different approaches
problem = (
    "A farmer has 3 fields. The first field is 2.5 acres and yields 180 bushels per acre. "
    "The second field is 4 acres and yields 150 bushels per acre. "
    "The third field is 1.8 acres and yields 210 bushels per acre. "
    "If wheat sells for $7.50 per bushel, what is the farmer's total revenue?"
)
# Correct: 2.5×180 + 4×150 + 1.8×210 = 450 + 600 + 378 = 1428 bushels × $7.50 = $10,710

# (a) Direct answer — no CoT, no thinking
msgs_direct = [{"role": "user", "content": f"Give the answer as a single number: {problem}"}]
direct_ans = generate(msgs_direct, temperature=0.1, thinking=False)
print("(a) DIRECT (no CoT, thinking=False):")
print(f"    Answer: {direct_ans}")

# (b) Prompt-induced CoT — thinking=False but prompt asks for steps
msgs_prompt_cot = [{"role": "user", "content": f"{problem}\n\nLet's think step by step."}]
prompt_cot_ans = generate(msgs_prompt_cot, temperature=0.6, thinking=False)
print("\n(b) PROMPT-INDUCED COT (thinking=False, CoT prompt):")
print(f"    Answer ({len(prompt_cot_ans.split())} words):")
print(f"    {prompt_cot_ans[:500]}")

# (c) Native thinking mode
msgs_native = [{"role": "user", "content": problem}]
thought, native_ans = generate(msgs_native, temperature=0.6)
print("\n(c) NATIVE THINKING (thinking=True):")
print(f"    Thinking ({len(thought.split())} words):")
print(f"    {thought[:500]}")
print(f"\n    Answer: {native_ans}")
print(f"\nCorrect answer: $10,710")

In [ ]:
# Inspect the native thinking trace more closely
# Look at: structure, self-corrections, verification steps

print("=== FULL NATIVE THINKING TRACE ===")
print(thought)
print("\n" + "="*50)

# Analyze the thinking trace
lines = thought.split("\n")
print(f"\nThinking trace statistics:")
print(f"  Total lines: {len(lines)}")
print(f"  Total words: {len(thought.split())}")

# Check for self-correction patterns
correction_markers = ["wait", "actually", "no,", "let me recalculate", "correction", "I made an error",
                      "that's wrong", "reconsider", "hmm", "mistake"]
corrections = [line.strip() for line in lines
               if any(marker in line.lower() for marker in correction_markers)]
if corrections:
    print(f"  Self-corrections found ({len(corrections)}):")
    for c in corrections[:5]:
        print(f"    → {c[:100]}")
else:
    print("  No explicit self-corrections detected")

# Check for verification
verification_markers = ["verify", "check", "confirm", "double-check", "sanity"]
verifications = [line.strip() for line in lines
                 if any(v in line.lower() for v in verification_markers)]
if verifications:
    print(f"  Verification steps ({len(verifications)}):")
    for v in verifications[:3]:
        print(f"    → {v[:100]}")

### Why Native Thinking Is Different

The difference between prompt-induced CoT and native thinking is **training**, not just prompting:

- **Prompt-induced CoT** relies on patterns the model learned during **pretraining** — it saw step-by-step solutions in textbooks and StackOverflow answers, so it can imitate them when asked. But it wasn't specifically *optimized* for reasoning quality.

- **Native thinking mode** (Qwen3's `<think>` blocks) was trained using **GRPO (Group Relative Policy Optimization)** — a reinforcement learning method where the model is rewarded for producing reasoning traces that lead to correct answers. The model learned *which kinds of reasoning* are most effective, not just how to format steps.

Key differences in practice:
1. **Self-correction**: Native thinking traces often contain "wait, let me reconsider..." moments — the model backtracks when it spots an error. Prompt-induced CoT rarely does this.
2. **Depth calibration**: The model allocates more thinking tokens to harder problems. Prompt-induced CoT tends to give similar-length responses regardless of difficulty.
3. **Answer quality**: The native thinking answer is typically cleaner because the messy reasoning is hidden in `<think>` — the user sees only the polished result.
4. **Token cost**: Native thinking uses significantly more tokens (the `<think>` block is often 3-10x longer than the answer).

## Inspecting the Thinking Trace: How the Model Allocates "Thought"

A fascinating property of native thinking mode is that **the model spends more tokens thinking about harder problems**. It "knows" when it needs to think more. Let's test this with problems of increasing difficulty.

In [ ]:
import time

# Three problems of increasing difficulty
problems = [
    {
        "label": "EASY — Simple arithmetic",
        "prompt": "What is 15 × 8?",
        "answer": "120"
    },
    {
        "label": "MEDIUM — Multi-step word problem",
        "prompt": (
            "A train leaves City A at 9:00 AM traveling at 80 km/h toward City B. "
            "Another train leaves City B at 10:00 AM traveling at 120 km/h toward City A. "
            "If the cities are 560 km apart, at what time do the trains meet?"
        ),
        "answer": "12:24 PM (or equivalent)"
    },
    {
        "label": "HARD — Logic puzzle",
        "prompt": (
            "Five houses in a row are painted different colors. In each house lives a person of a "
            "different nationality. Each person drinks a different beverage, smokes a different brand, "
            "and keeps a different pet.\n"
            "Clues:\n"
            "1. The Brit lives in the red house.\n"
            "2. The Swede keeps dogs.\n"
            "3. The Dane drinks tea.\n"
            "4. The green house is immediately to the left of the white house.\n"
            "5. The green house owner drinks coffee.\n"
            "6. The person who smokes Pall Mall keeps birds.\n"
            "7. The owner of the yellow house smokes Dunhill.\n"
            "8. The person in the center house drinks milk.\n"
            "9. The Norwegian lives in the first house.\n"
            "10. The Blend smoker has a neighbor who keeps cats.\n"
            "11. The person who keeps horses lives next to the Dunhill smoker.\n"
            "12. The Blue Master smoker drinks beer.\n"
            "13. The German smokes Prince.\n"
            "14. The Norwegian lives next to the blue house.\n"
            "15. The Blend smoker has a neighbor who drinks water.\n"
            "Who owns the fish?"
        ),
        "answer": "The German"
    }
]

results = []
for p in problems:
    msgs = [{"role": "user", "content": p["prompt"]}]
    t0 = time.time()
    thought, answer = generate(msgs, max_new_tokens=4096, temperature=0.6)
    elapsed = time.time() - t0

    think_words = len(thought.split()) if thought else 0
    answer_words = len(answer.split()) if answer else 0

    results.append({
        "label": p["label"],
        "think_words": think_words,
        "answer_words": answer_words,
        "ratio": round(think_words / max(answer_words, 1), 1),
        "time": round(elapsed, 1),
        "thought": thought,
        "answer": answer,
        "expected": p["answer"]
    })

    print(f"\n{'='*60}")
    print(f"{p['label']}")
    print(f"{'='*60}")
    print(f"Thinking: {think_words} words | Answer: {answer_words} words | Ratio: {results[-1]['ratio']}x | Time: {elapsed:.1f}s")
    print(f"\nThinking trace (first 400 chars):")
    print(thought[:400] if thought else "(empty)")

    # Check for self-corrections
    correction_markers = ["wait", "actually", "no,", "let me recalculate", "reconsider", "hmm", "mistake"]
    corrections = [line.strip() for line in thought.split("\n")
                   if any(m in line.lower() for m in correction_markers)]
    if corrections:
        print(f"\nSelf-corrections found: {len(corrections)}")
        for c in corrections[:3]:
            print(f"  → {c[:120]}")

    print(f"\nAnswer: {answer[:300]}")
    print(f"Expected: {p['expected']}")

In [ ]:
# Summary table: thinking scales with difficulty
print("\n" + "="*70)
print(f"{'Problem':<35} {'Think Words':>12} {'Answer Words':>13} {'Ratio':>7} {'Time':>7}")
print("-"*70)
for r in results:
    print(f"{r['label']:<35} {r['think_words']:>12} {r['answer_words']:>13} {r['ratio']:>6}x {r['time']:>6}s")
print("="*70)
print("\nKey insight: The thinking trace grows with problem difficulty.")
print("The model dynamically allocates more 'thought' to harder problems.")

### What We Observe

The thinking-to-answer ratio is a rough proxy for how "hard" the model finds a problem:

- **Simple arithmetic**: Very short thinking trace (sometimes just restating the problem), ratio near 1-2x
- **Multi-step word problems**: Moderate thinking with clear sequential steps, ratio around 3-5x
- **Logic puzzles**: Extensive thinking with backtracking, hypothesis testing, and verification, ratio often 5-15x

This adaptive behavior emerges from GRPO training — the model learned that allocating more reasoning tokens to harder problems leads to better rewards. It's a form of **learned metacognition**: the model has an implicit sense of problem difficulty and adjusts its computational effort accordingly.

Also look for **self-correction patterns** in the harder problems. Phrases like "wait, actually..." or "let me reconsider..." indicate the model is re-evaluating its reasoning mid-trace. This is one of the most powerful features of native thinking — the model can catch and fix its own mistakes before committing to a final answer.

## 3 · Few-Shot CoT — Worked Examples

Provide one or more **worked examples** with explicit step-by-step reasoning. The model learns the reasoning pattern from the examples and applies it to the new question.

In [ ]:
few_shot_cot = [
    {"role": "system", "content": "You are a helpful math tutor. Solve each problem step by step, then state the final answer."},
    # Worked example 1
    {"role": "user", "content": "If a car drives 60 km in 1.5 hours, what is its average speed?"},
    {"role": "assistant", "content": (
        "Step 1: Identify the values. Distance = 60 km, Time = 1.5 hours.\n"
        "Step 2: Apply the formula. Average speed = Distance / Time = 60 / 1.5.\n"
        "Step 3: Calculate. 60 / 1.5 = 40.\n"
        "Final answer: The average speed is 40 km/h."
    )},
    # Worked example 2
    {"role": "user", "content": "A cyclist covers 25 km in 50 minutes. What is the average speed in km/h?"},
    {"role": "assistant", "content": (
        "Step 1: Convert time to hours. 50 minutes = 50/60 = 5/6 hours ≈ 0.8333 hours.\n"
        "Step 2: Apply the formula. Average speed = 25 / (5/6) = 25 × 6/5.\n"
        "Step 3: Calculate. 25 × 6/5 = 30.\n"
        "Final answer: The average speed is 30 km/h."
    )},
    # Actual question
    {"role": "user", "content": "A car travels 150 km at 60 km/h, then another 100 km at 50 km/h. What is the average speed for the entire journey?"}
]

print("Few-shot CoT:")
thought, answer = generate(few_shot_cot, temperature=0.6)
print("Thinking:", thought[:300] if thought else '(none)')
print("Answer:", answer)

## 4 · Structured CoT — Numbered Steps with Explicit Format

For complex problems, give the model an explicit **numbered-step template** so it follows a systematic process: state what to calculate, write the formula, compute, explain.

In [ ]:
complex_question = (
    "A car travels 150 km at 60 km/h, then another 100 km at 50 km/h. "
    "What is the average speed for the entire journey?"
)

structured_prompt = f"""Solve the following problem step by step. For each step:
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {complex_question}

Solution:"""

structured_cot = [{"role": "user", "content": structured_prompt}]
print("Structured CoT:")
thought, answer = generate(structured_cot, temperature=0.6)
print("Thinking:", thought[:400] if thought else '(none)')
print("Answer:", answer)

## 5 · Comparative Analysis — Standard vs CoT on a Challenging Problem

Let's compare standard prompting and CoT prompting on a harder, multi-step geometry/unit-conversion problem. Standard prompting often gives a wrong or vague number, while CoT shows (and usually gets right) every intermediate calculation.

In [ ]:
challenging_question = (
    "A cylindrical water tank with a radius of 1.5 meters and a height of 4 meters is 2/3 full. "
    "If water is being added at a rate of 10 liters per minute, how long will it take for the tank to overflow? "
    "Give your answer in hours and minutes, rounded to the nearest minute. "
    "(Use 3.14159 for π and 1000 liters = 1 cubic meter)"
)

# --- Standard prompt (no CoT) ---
standard = [{"role": "user", "content": f"Answer the following question concisely: {challenging_question}"}]
print("Standard Response (no thinking):")
print(generate(standard, temperature=0.1, thinking=False))

# --- Structured CoT prompt ---
cot_prompt = f"""Solve the following problem step by step. For each step:
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {challenging_question}

Solution:"""

cot = [{"role": "user", "content": cot_prompt}]
print("\nChain of Thought Response (with native thinking):")
thought, answer = generate(cot, temperature=0.6)
print("Thinking:", thought[:500] if thought else '(none)')
print("Answer:", answer)

## When CoT Hurts: Simple Tasks and the Overthinking Problem

Chain of Thought is not universally beneficial. For simple, high-confidence tasks, CoT can actually **hurt** performance by:
1. **Adding latency** — more tokens to generate means slower responses
2. **Introducing errors** — overthinking a simple question can lead to second-guessing the correct answer
3. **Wasting compute** — CoT on trivial questions is like using calculus to add 2+2

In [ ]:
# CoT on trivial tasks — does it help or hurt?
simple_questions = [
    "What color is the sky on a clear day?",
    "What is 2 + 2?",
    "What is the capital of France?",
    "Is water wet?",
]

print("=== SIMPLE TASKS: Direct vs. CoT ===\n")
for q in simple_questions:
    # Direct answer (no thinking)
    msgs_direct = [{"role": "user", "content": q}]
    direct = generate(msgs_direct, temperature=0.1, thinking=False, max_new_tokens=100)

    # With native thinking
    thought, answer = generate(msgs_direct, temperature=0.6, max_new_tokens=512)

    think_words = len(thought.split()) if thought else 0
    print(f"Q: {q}")
    print(f"  Direct (no thinking): {direct[:100]}")
    print(f"  With thinking ({think_words} words of thought): {answer[:100]}")
    print()

In [ ]:
# Token count comparison: simple vs complex tasks with and without thinking
import time

test_cases = [
    ("Simple: 2+3", "What is 2 + 3?"),
    ("Complex: multi-step", (
        "A store has a 25% off sale. An item originally costs $80. "
        "You also have a $10 coupon applied after the discount. "
        "Sales tax is 8%. What is the final price?"
    )),
]

print(f"{'Task':<25} {'Mode':<20} {'Think Words':>12} {'Answer Words':>13} {'Time':>7}")
print("-" * 80)

for label, question in test_cases:
    msgs = [{"role": "user", "content": question}]

    # Without thinking
    t0 = time.time()
    direct = generate(msgs, temperature=0.1, thinking=False, max_new_tokens=512)
    t_direct = time.time() - t0
    print(f"{label:<25} {'No thinking':<20} {'—':>12} {len(direct.split()):>13} {t_direct:>6.1f}s")

    # With thinking
    t0 = time.time()
    thought, answer = generate(msgs, temperature=0.6, max_new_tokens=1024)
    t_think = time.time() - t0
    think_words = len(thought.split()) if thought else 0
    print(f"{label:<25} {'Native thinking':<20} {think_words:>12} {len(answer.split()):>13} {t_think:>6.1f}s")
    print()

print("Key observation: For simple tasks, thinking mode adds significant overhead")
print("with little to no accuracy benefit.")

### The CoT Trade-off

| Factor | Simple Tasks | Complex Tasks |
|--------|-------------|---------------|
| **Accuracy gain from CoT** | None or negative | Significant |
| **Latency cost** | High relative to task value | Worth it |
| **Token cost** | Wasted | Justified |
| **Risk of overthinking** | High — model may second-guess correct intuition | Low — extra thought helps |

**Rule of thumb**: Use CoT when the problem requires **multiple reasoning steps** or when the model's first-instinct accuracy is below ~90%. For factual recall, simple arithmetic, or classification tasks, direct prompting is faster and equally accurate.

In production systems, this suggests a **routing strategy**: classify incoming queries by complexity and only activate CoT/thinking mode for queries that benefit from it. This saves both latency and API costs.

## 6 · Problem-Solving Application — Logical Reasoning Puzzle

CoT is especially powerful for **logical reasoning** tasks where the model must systematically enumerate possibilities and eliminate contradictions. Here we give the model a detailed analysis framework.

In [ ]:
logical_puzzle = (
    "In a room, there are three people: Amy, Bob, and Charlie. "
    "One of them always tells the truth, one always lies, and one alternates between truth and lies. "
    "Amy says, 'Bob is a liar.' "
    "Bob says, 'Charlie alternates between truth and lies.' "
    "Charlie says, 'Amy and I are both liars.' "
    "Determine the nature (truth-teller, liar, or alternator) of each person."
)

reasoning_framework = f"""Analyze the following logical puzzle thoroughly. Follow these steps in your analysis:

1. List the Facts: Summarize all given information and identify all characters.
2. Identify Possible Roles: Determine all possible roles (truth-teller, liar, alternator).
3. Note the Constraints: Outline any rules or relationships specified in the puzzle.
4. Generate Possible Scenarios: Systematically consider all possible role assignments.
5. Test Each Scenario: For each scenario, check every statement for consistency.
6. Eliminate Inconsistent Scenarios: Discard any that lead to contradictions.
7. Conclude the Solution: Identify the remaining consistent scenario.
8. Provide a Clear Answer: State the role of each person and explain why.

Scenario:

{logical_puzzle}

Analysis:"""

messages = [{"role": "user", "content": reasoning_framework}]
thought, answer = generate(messages, max_new_tokens=2048, temperature=0.6)
print("=== THINKING ===")
print(thought[:800] if thought else '(none)')
print("\n=== ANSWER ===")
print(answer)

## Expanded Examples: CoT Across Diverse Domains

Chain of Thought is not just for math. Let's explore how CoT reasoning works across fundamentally different types of problems — each demanding a different style of reasoning.

In [ ]:
# EXAMPLE 1: Mathematical Proof
# Tests: formal logical deduction, ability to construct valid arguments

proof_prompt = (
    "Prove that the square root of 2 is irrational. "
    "Present a rigorous proof by contradiction, showing each step clearly."
)

msgs = [{"role": "user", "content": proof_prompt}]
thought, answer = generate(msgs, max_new_tokens=2048, temperature=0.6)

print("=== EXAMPLE 1: Mathematical Proof (√2 is irrational) ===")
print(f"\nThinking trace ({len(thought.split())} words):")
print(thought[:600] if thought else "(empty)")
print(f"\nProof:")
print(answer[:800])

In [ ]:
# EXAMPLE 2: Logic Puzzle (different from the one already in the notebook)
# Tests: constraint satisfaction, systematic enumeration

logic_prompt = (
    "Three friends — Alice, Bob, and Carol — each ordered a different drink "
    "(coffee, tea, juice) and a different dessert (cake, pie, ice cream).\n\n"
    "Clues:\n"
    "1. Alice didn't order coffee or cake.\n"
    "2. Bob ordered tea.\n"
    "3. The person who ordered coffee also ordered cake.\n"
    "4. Carol didn't order ice cream.\n\n"
    "Determine what each person ordered."
)

msgs = [{"role": "user", "content": logic_prompt}]
thought, answer = generate(msgs, max_new_tokens=2048, temperature=0.6)

print("=== EXAMPLE 2: Logic Puzzle (Constraint Satisfaction) ===")
print(f"\nThinking trace ({len(thought.split())} words):")
print(thought[:600] if thought else "(empty)")
print(f"\nSolution:")
print(answer[:500])
print(f"\nExpected: Bob=tea+cake? Let's check — Clue 2: Bob has tea. Clue 3: coffee→cake.")
print("So coffee+cake go together (not Bob). Clue 1: Alice has no coffee, no cake.")
print("So Carol has coffee+cake. Alice has juice. Alice has no cake → pie or ice cream.")
print("Clue 4: Carol has no ice cream (she has cake). So Alice gets ice cream or pie.")
print("Remaining desserts for Alice & Bob: pie and ice cream. No further constraint → Alice=juice+pie or juice+ice cream.")

In [ ]:
# EXAMPLE 3: Code Debugging
# Tests: program tracing, identifying logical errors, fix reasoning

debug_prompt = """The following Python function is supposed to return the second largest
number in a list, but it has a bug. Find the bug and fix it.

def second_largest(nums):
    if len(nums) < 2:
        return None
    first = second = float('-inf')
    for n in nums:
        if n > first:
            second = first
            first = n
        elif n > second:
            second = n
    return second

# Test case that fails:
# second_largest([5, 5, 5]) returns -inf instead of 5
# second_largest([1, 2]) returns 1 (correct)
# second_largest([3, 1, 2]) returns 2 (correct)

Explain the bug, why it happens, and provide the corrected code."""

msgs = [{"role": "user", "content": debug_prompt}]
thought, answer = generate(msgs, max_new_tokens=2048, temperature=0.6)

print("=== EXAMPLE 3: Code Debugging ===")
print(f"\nThinking trace ({len(thought.split())} words):")
print(thought[:600] if thought else "(empty)")
print(f"\nAnalysis and fix:")
print(answer[:800])

In [ ]:
# EXAMPLE 4: Ethical Dilemma
# Tests: multi-perspective reasoning, weighing trade-offs, nuanced analysis

ethics_prompt = (
    "A self-driving car's brakes fail. It can either:\n"
    "(A) Stay on course and hit 3 elderly pedestrians crossing legally, or\n"
    "(B) Swerve onto the sidewalk and hit 1 young child.\n\n"
    "Analyze this dilemma from multiple ethical frameworks: "
    "utilitarian, deontological, virtue ethics, and care ethics. "
    "What are the strongest arguments for each choice? "
    "Do NOT give a single 'right answer' — instead, show the reasoning "
    "each framework uses and where they disagree."
)

msgs = [{"role": "user", "content": ethics_prompt}]
thought, answer = generate(msgs, max_new_tokens=2048, temperature=0.6)

print("=== EXAMPLE 4: Ethical Dilemma (Multi-Framework Analysis) ===")
print(f"\nThinking trace ({len(thought.split())} words):")
print(thought[:600] if thought else "(empty)")
print(f"\nAnalysis:")
print(answer[:1000])

### Reasoning Style Varies by Domain

Across these four examples, notice how the **structure of reasoning** differs:

| Domain | Reasoning style | CoT characteristics |
|--------|----------------|-------------------|
| **Mathematical proof** | Deductive, linear | Each step follows logically from the previous; formal structure |
| **Logic puzzle** | Constraint propagation | Systematic enumeration, elimination, backtracking |
| **Code debugging** | Program tracing | Mental execution, state tracking, hypothesis testing |
| **Ethical dilemma** | Multi-perspective, dialectical | Framework comparison, weighing trade-offs, no single answer |

This diversity is why CoT is so powerful — it's not a single reasoning strategy but a **framework for externalizing whatever reasoning style the problem demands**. The model adapts its thinking trace structure to match the domain.

### Key Takeaways from This Tutorial

1. **CoT works by decomposing hard predictions into easy ones** — each step is within the model's capability, even if the overall problem isn't.

2. **Native thinking mode > prompt-induced CoT** — models trained with RL for reasoning (like Qwen3) produce higher-quality, self-correcting traces.

3. **Thinking scales with difficulty** — the model allocates more reasoning tokens to harder problems (learned metacognition).

4. **CoT is not free** — it adds latency and tokens. Use it when the problem genuinely requires multi-step reasoning.

5. **Different problems need different reasoning** — CoT is a general framework that adapts to deductive, abductive, constraint-based, and dialectical reasoning styles.